# 3장 02. MLflow container 기록과 model URI 확인

이 notebook은 2장에서 만든 모델 평가 기록이 MLflow 기록과 배포 후보 URI로 어떻게 이어지는지 확인합니다. MLflow 서버를 container로 띄우고 새 평가 기록을 남기는 실행은 `demos/ch02_mlflow/01_run_with_docker_mlflow.sh`가 담당합니다. 이 notebook에서는 그 결과를 읽고, KServe manifest가 어떤 모델 후보를 가리키는지 확인합니다.


## 기본 개념: MLflow Tracking, Model, Registry, URI

MLflow Tracking은 학습이나 평가 실행을 기록하는 도구입니다. 어떤 데이터셋을 썼는지, threshold는 얼마였는지, precision과 recall은 얼마였는지, 어떤 artifact가 남았는지를 한 run 안에 모읍니다. 수업에서는 MLflow 자체도 container로 띄워서 “모델 실행 기록을 서버에 남긴다”는 흐름을 보여 줍니다.

MLflow Model은 모델 파일 하나보다 넓은 배포 단위입니다. 모델 artifact, flavor, dependency, signature 같은 실행 정보를 함께 담을 수 있습니다. 그래서 MLOps에서는 단순한 `.pkl` 파일보다 MLflow Model package나 registry version을 기준으로 배포 후보를 설명하는 편이 안전합니다.

Model Registry는 등록된 모델의 version과 alias를 관리합니다. `candidate` alias는 다음 배포 후보를 의미하고, `champion` 또는 운영 alias는 현재 운영 모델을 의미할 수 있습니다. alias를 사용하면 코드나 manifest가 version number를 직접 박아두지 않고 운영 의미를 가진 포인터를 참조할 수 있습니다.

`models:/risk-classifier@candidate`는 MLflow registry 안의 `risk-classifier` 모델 중 `candidate` alias가 붙은 버전을 뜻합니다. Kubernetes 배포에서는 실제 artifact 저장 위치가 필요하므로, 이 실습에서는 KServe manifest의 `storageUri`도 함께 확인합니다.


In [1]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas


## 1. 평가 기록과 manifest 파일 찾기

MLflow 배포 기준은 모델 파일 하나가 아니라 평가 기록, 모델 이름, version, alias, artifact 위치가 연결되어 있는지 확인하는 것입니다. 이 셀은 로컬 경로와 JupyterLite 경로 중 실제 존재하는 파일을 찾습니다.


In [2]:
from pathlib import Path
import json
import re

metric_paths = [
    Path("artifacts/experiments/chapter_02/model_test_eval.json"),
    Path("../artifacts/experiments/chapter_02/model_test_eval.json"),
    Path("../../artifacts/experiments/chapter_02/model_test_eval.json"),
]
manifest_paths = [
    Path("gitops/base/inferenceservice.yaml"),
    Path("../../gitops/base/inferenceservice.yaml"),
    Path("artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
    Path("../artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
]

metric_path = next(path for path in metric_paths if path.exists())
manifest_path = next(path for path in manifest_paths if path.exists())

metric_record = json.loads(metric_path.read_text())
manifest_text = manifest_path.read_text()

print("metric record:", metric_path)
print("serving manifest:", manifest_path)


metric record: ../../artifacts/experiments/chapter_02/model_test_eval.json
serving manifest: ../../gitops/base/inferenceservice.yaml


## 2. MLflow URI와 storage URI 확인

`models:/risk-classifier@candidate`는 수강생이 외울 문자열이 아니라, 어떤 registered model의 어떤 alias를 serving 후보로 쓰는지 보여 주는 주소입니다. KServe의 `storageUri`는 실제 runtime이 모델 artifact를 읽을 저장 위치입니다.


In [3]:
mlflow_uri = re.search(r"ai-quality/mlflow-model-uri:\s*(\S+)", manifest_text).group(1)
storage_uri = re.search(r"storageUri:\s*(\S+)", manifest_text).group(1)
registered_model, alias = re.search(r"models:/([^@]+)@(.+)", mlflow_uri).groups()

pd.DataFrame([
    {"field": "mlflow_model_uri", "value": mlflow_uri},
    {"field": "registered_model", "value": registered_model},
    {"field": "alias", "value": alias},
    {"field": "kserve_storage_uri", "value": storage_uri},
])


,field,value
0,mlflow_model_uri,models:/risk-classifier@candidate
1,registered_model,risk-classifier
2,alias,candidate
3,kserve_storage_uri,pvc://ai-quality-models/artifacts/risk-classif...


## 3. 2장 평가 기록과 배포 후보 연결

이 셀은 2장에서 기록한 test metric과 3장에서 배포할 candidate URI를 같은 표로 묶습니다. 여기서 확인할 것은 모델 성능을 다시 계산하는 일이 아니라, serving 후보를 설명할 수 있는 평가 조건이 남아 있는지입니다.


In [4]:
params = metric_record["params"]
metrics = metric_record["metrics"]

pd.DataFrame([
    {"check": "dataset", "value": params["dataset_name"]},
    {"check": "model_version", "value": params["model_version"]},
    {"check": "threshold", "value": params["operating_threshold"]},
    {"check": "precision", "value": round(metrics["precision"], 4)},
    {"check": "recall", "value": round(metrics["recall"], 4)},
    {"check": "false_positive", "value": int(metrics["false_positive"])},
    {"check": "false_negative", "value": int(metrics["false_negative"])},
    {"check": "serving_candidate", "value": mlflow_uri},
])


,check,value
0,dataset,vital_signs_test
1,model_version,v1
2,threshold,0.5
3,precision,1.0
4,recall,0.4266
5,false_positive,0
6,false_negative,5989
7,serving_candidate,models:/risk-classifier@candidate


## 4. 이 단계에서 말할 수 있는 것

이 출력으로 말할 수 있는 것은 candidate URI와 평가 기록의 연결입니다. MLflow container에서 기록한 run이 있고, 그 기록의 metric과 artifact가 배포 후보 설명에 쓰일 수 있다는 뜻입니다.

아직 FastAPI 응답, Argo CD sync, KServe Ready, endpoint 응답은 확인하지 않았습니다. 다음 단계에서는 FastAPI/Compose로 API serving을 확인하고, 그 뒤 Kubernetes에서 MLflow manifest와 KServe manifest를 순서대로 확인합니다.
